In [ ]:
import os, json, subprocess, sys, time, glob, shutil
print('python', sys.version.split()[0])
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:1500])
gpu = subprocess.run(['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
total_mem = sum(int(line.split()[0].replace(',', '')) for line in gpu if line)
print('GPU count:', len(gpu), '| total VRAM MiB:', total_mem)
os.environ['TOTAL_VRAM_MIB'] = str(total_mem)


In [ ]:
os.chdir('/kaggle/working')
r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mattdani21/ModelSwapper.git'], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
os.chdir('/kaggle/working/ModelSwapper')
print(subprocess.run(['git', 'log', '-1', '--format=%h %ci'], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'pytest'], capture_output=True, text=True)
print('pytest install rc:', r.returncode)


In [ ]:
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/llama.cpp/build/bin/llama-server'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git'], capture_output=True, text=True, check=True)
    r = subprocess.run(['cmake', '-B', '/kaggle/working/llama.cpp/build', '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release', '-S', '/kaggle/working/llama.cpp'], capture_output=True, text=True)
    print('cmake configure rc:', r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
    r = subprocess.run(['cmake', '--build', '/kaggle/working/llama.cpp/build', '--config', 'Release', '-j', '4', '--target', 'llama-server'], capture_output=True, text=True)
    print('cmake build rc:', r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
print('llama-server exists:', os.path.exists('/kaggle/working/llama.cpp/build/bin/llama-server'))
os.environ['LLAMA_SERVER'] = '/kaggle/working/llama.cpp/build/bin/llama-server'
os.environ['PATH'] = '/kaggle/working/llama.cpp/build/bin:' + os.environ['PATH']


In [ ]:
os.chdir('/kaggle/working')
total_mem = int(os.environ.get('TOTAL_VRAM_MIB', '0'))
MODELS = {
    'reason': 'https://huggingface.co/Qwen/Qwen3-14B-GGUF/resolve/main/Qwen3-14B-Q4_K_M.gguf',
    'code_q4': 'https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF/resolve/main/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf',
    'code_q3': 'https://huggingface.co/unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF/resolve/main/Qwen3-Coder-30B-A3B-Instruct-Q3_K_M.gguf',
}
os.makedirs('/kaggle/working/models', exist_ok=True)
def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1e9:
        print('cached', dest); return
    r = subprocess.run(['wget', '-q', '-O', dest, url], capture_output=True, text=True, timeout=3600)
    print('wget', os.path.basename(dest), 'rc:', r.returncode, 'size:', os.path.getsize(dest) // 1e6, 'MB')
fetch(MODELS['reason'], '/kaggle/working/models/Qwen3-14B-Q4_K_M.gguf')
code_url = MODELS['code_q4'] if total_mem >= 30000 else MODELS['code_q3']
code_name = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf' if total_mem >= 30000 else 'Qwen3-Coder-30B-A3B-Instruct-Q3_K_M.gguf'
print('choosing coder quant:', code_name)
fetch(code_url, '/kaggle/working/models/' + code_name)
MODEL_PATHS = {
    'reason': '/kaggle/working/models/Qwen3-14B-Q4_K_M.gguf',
    'code': '/kaggle/working/models/' + code_name,
    'review': '/kaggle/working/models/Qwen3-14B-Q4_K_M.gguf',
}
print(MODEL_PATHS)


In [ ]:
os.chdir('/kaggle/working/ModelSwapper')
env = dict(os.environ)
env['LLAMA_CONTEXT'] = '4096'
env['LLAMA_NGPU'] = '99'
models_json = json.dumps(MODEL_PATHS)
cmd = [sys.executable, 'pipeline/run_pipeline.py',
       '--models-json', models_json,
       '--out', '/kaggle/working/pipeline-results.json',
       '--capsule-dir', '/kaggle/working/capsules',
       '--port-base', '8950',
       '--max-iterations', '3',
       '--max-tokens', '2048']
print('running:', ' '.join(cmd[:6]), '...')
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=8 * 3600)
print('pipeline rc:', r.returncode, '| wall:', round((time.time() - t0) / 60, 1), 'min')
print((r.stdout or '')[-2500:])
print((r.stderr or '')[-1000:])


In [ ]:
os.chdir('/kaggle/working')
res_path = '/kaggle/working/pipeline-results.json'
if os.path.exists(res_path):
    d = json.load(open(res_path))
    print('PASS RATE:', d.get('pass_rate'), '|', d.get('tasks_passed'), '/', d.get('tasks_total'))
    print('per_category:', d.get('per_category'))
    print('mean_wall_clock_s:', d.get('mean_wall_clock_s'))
    print('mean_load_s:', d.get('mean_load_s'), '| mean_evict_s:', d.get('mean_evict_s'))
    with open('/kaggle/working/pipeline-summary.txt', 'w') as f:
        f.write(json.dumps({k: v for k, v in d.items() if k != 'results'}, indent=2))
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working', 'pipeline-results.json')
shutil.make_archive('/kaggle/working/capsules', 'zip', '/kaggle/working/capsules')
print('outputs:', [f for f in glob.glob('/kaggle/working/*.zip') + glob.glob('/kaggle/working/pipeline-*') + glob.glob('/kaggle/working/capsules-*')])
